# Notebook 03 — Model Setup and Tokenizer

Validates:
- GPU availability and VRAM
- Qwen2.5-3B-Instruct tokenizer and chat template
- Model loading in BF16
- Token count for sample inputs

In [1]:
import sys
import subprocess
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

sys.path.insert(0, str(Path('..').resolve()))
from src.model_utils import load_model_and_tokenizer, print_model_info
from src.data_utils import SYSTEM_PROMPT_MOD2SHAK, SYSTEM_PROMPT_SHAK2MOD

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
HF_CACHE = Path('..') / 'data' / 'raw' / 'huggingface_cache'

print(f'Model: {MODEL_ID}')

Model: Qwen/Qwen2.5-3B-Instruct


## 1. GPU / Hardware Check

In [2]:
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU:             {gpu.name}')
    print(f'Total VRAM:      {gpu.total_memory / 1e9:.1f} GB')
    print(f'CUDA version:    {torch.version.cuda}')
    print(f'PyTorch version: {torch.__version__}')

# nvidia-smi snapshot
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free,memory.used',
         '--format=csv,noheader,nounits'],
        capture_output=True, text=True
    )
    print('\nnvidia-smi output:')
    print(result.stdout)
except FileNotFoundError:
    print('nvidia-smi not found (non-NVIDIA or headless environment)')

CUDA available: True
GPU:             NVIDIA GeForce RTX 5070 Laptop GPU
Total VRAM:      8.5 GB
CUDA version:    12.8
PyTorch version: 2.11.0+cu128

nvidia-smi output:
NVIDIA GeForce RTX 5070 Laptop GPU, 8151, 6677, 1215



## 2. Load Tokenizer

In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side='right',
    cache_dir=str(HF_CACHE),
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Vocab size:               {tokenizer.vocab_size:,}')
print(f'EOS token:                {tokenizer.eos_token!r}')
print(f'PAD token:                {tokenizer.pad_token!r}')
print(f'Chat template available:  {tokenizer.chat_template is not None}')
print(f'Padding side:             {tokenizer.padding_side}')

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

C:\Users\pra73\.conda\envs\advnlp\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pra73\Desktop\Projects\Active_Gits\Mormon-NLT\data\raw\huggingface_cache\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size:               151,643
EOS token:                '<|im_end|>'
PAD token:                '<|endoftext|>'
Chat template available:  True
Padding side:             right


## 3. Verify Chat Template

In [4]:
# Test mod→shak direction
sample_messages_m2s = [
    {'role': 'system',    'content': SYSTEM_PROMPT_MOD2SHAK},
    {'role': 'user',      'content': 'I have half a mind to hit you.'},
    {'role': 'assistant', 'content': 'I have a mind to strike thee.'},
]

formatted_train = tokenizer.apply_chat_template(
    sample_messages_m2s,
    tokenize=False,
    add_generation_prompt=False,  # training: include the assistant reply
)
formatted_infer = tokenizer.apply_chat_template(
    sample_messages_m2s[:2],      # system + user only
    tokenize=False,
    add_generation_prompt=True,   # inference: prompt the assistant
)

print('=== Training format (includes assistant reply) ===\n')
print(formatted_train)
print('\n=== Inference format (add_generation_prompt=True) ===\n')
print(formatted_infer)

=== Training format (includes assistant reply) ===

<|im_start|>system
You are an expert translator of Modern English into Shakespearean English. Translate the following modern English text into authentic Shakespearean style, preserving the meaning while using appropriate Early Modern English vocabulary, grammar, and poetic diction.<|im_end|>
<|im_start|>user
I have half a mind to hit you.<|im_end|>
<|im_start|>assistant
I have a mind to strike thee.<|im_end|>


=== Inference format (add_generation_prompt=True) ===

<|im_start|>system
You are an expert translator of Modern English into Shakespearean English. Translate the following modern English text into authentic Shakespearean style, preserving the meaning while using appropriate Early Modern English vocabulary, grammar, and poetic diction.<|im_end|>
<|im_start|>user
I have half a mind to hit you.<|im_end|>
<|im_start|>assistant



In [5]:
# Token counts for sample inputs at various lengths
test_inputs = [
    'What is your name?',
    'I love you more than words can say.',
    'She is the most beautiful woman I have ever seen in all my years upon this earth.',
]

print(f'{"Text":<65} {"Chars":>6} {"Tokens":>8}')
print('-' * 82)
for text in test_inputs:
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT_MOD2SHAK},
            {'role': 'user',   'content': text}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    n_tokens = len(tokenizer.encode(prompt))
    print(f'{text[:62]:<65} {len(text):>6} {n_tokens:>8}')

Text                                                               Chars   Tokens
----------------------------------------------------------------------------------
What is your name?                                                    18       61
I love you more than words can say.                                   35       65
She is the most beautiful woman I have ever seen in all my yea        81       74


## 4. Load Base Model (BF16) — Smoke Test

In [6]:
# NOTE: This will download ~6GB on first run
print(f'Loading {MODEL_ID} in BF16 ...')

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    cache_dir=str(HF_CACHE),
)
model.config.use_cache = False

print_model_info(model)

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved  = torch.cuda.memory_reserved(0) / 1e9
    print(f'\nGPU memory allocated: {allocated:.2f} GB')
    print(f'GPU memory reserved:  {reserved:.2f} GB')

Loading Qwen/Qwen2.5-3B-Instruct in BF16 ...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Total parameters:     3,085,938,688
Trainable parameters: 3,085,938,688  (100.00%)

GPU memory allocated: 6.17 GB
GPU memory reserved:  6.25 GB


## 5. Zero-Shot Translation (Baseline)

In [7]:
model.eval()
sys.path.insert(0, str(Path('..').resolve()))
from src.inference import translate

zero_shot_tests = [
    ('What is your name?',                     'mod2shak'),
    ('I love you more than words can say.',    'mod2shak'),
    ('Thou art the very pink of courtesy.',   'shak2mod'),
    ('To thine own self be true.',            'shak2mod'),
]

print('=== Zero-Shot Translations (base model, no fine-tuning) ===\n')
for text, direction in zero_shot_tests:
    output = translate(model, tokenizer, text, direction=direction, max_new_tokens=128)
    label  = 'Mod→Shak' if direction == 'mod2shak' else 'Shak→Mod'
    print(f'[{label}] Source:  {text}')
    print(f'          Output:  {output}')
    print()

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Zero-Shot Translations (base model, no fine-tuning) ===

[Mod→Shak] Source:  What is your name?
          Output:  What is thy name, good friend?

[Mod→Shak] Source:  I love you more than words can say.
          Output:  I love thee more Than tongue may utter or desire express.

[Shak→Mod] Source:  Thou art the very pink of courtesy.
          Output:  You are the very epitome of courtesy.

[Shak→Mod] Source:  To thine own self be true.
          Output:  Be true to yourself.



In [8]:
# Free model memory before training notebooks
import gc
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Model unloaded. GPU memory freed.')

Model unloaded. GPU memory freed.
